#### LLaMA 2 with 7 BIllion parameter

In [1]:
from datasets import load_dataset, DatasetDict

# Load the JSONL file
train_ds = load_dataset("json", data_files="/content/finance_pc_train.jsonl", split="train")

# display the first row
print(train_ds[0])

# Check the column names
print(train_ds.column_names)


{'question': "What services does Iron Mountain provide to protect organizations' information and reduce storage costs?", 'context': 'Iron Mountain helps organizations protect their information and reduce storage costs by storing physical records and data backup media, offering information management solutions, and providing data center space.', 'answer': 'Iron Mountain provides services such as storing physical records and data backup media, offering information management solutions, and providing data center space for enterprise-class colocation and hyperscale deployments.'}
['question', 'context', 'answer']


In [2]:
full_ds = load_dataset("json", data_files="/content/finance_pc_train.jsonl", split="train")

# Split 10% for validation and 90% for the training
split_ds = full_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
val_ds = split_ds["test"]

print("Train samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Columns:", train_ds.column_names)
print("Sample:", train_ds[0])

Train samples: 900
Validation samples: 100
Columns: ['question', 'context', 'answer']
Sample: {'question': 'What is the significance of Note 13 in the context of legal proceedings described in the Annual Report on Form 10-K?', 'context': 'For a description of our significant pending legal proceedings, see Note 13 titled Commitments and Contingencies - Legal Proceedings of the Notes to Consolidated Financial Statements included in Part II, Item 8 of this Annual Report on Form 10-K.', 'answer': "Note 13 is significant because it contains a detailed description of the company's significant pending legal proceedings."}


In [3]:
from transformers import AutoTokenizer
from datasets import load_dataset
# load the model
model_name = "meta-llama/Llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load dataset and split
full_ds = load_dataset("json", data_files="/content/finance_pc_train.jsonl", split="train")
split_ds = full_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
val_ds = split_ds["test"]

# Tokenization with teacher forcing
def tokenize_qa(example):
    # Full prompt
    prompt_text = f"Question: {example['question']}\nContext: {example['context']}\nAnswer: "
    answer_text = example['answer'] + tokenizer.eos_token  # add EOS

    # Encode prompt
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=512)["input_ids"]
    # Encode answer
    answer_ids = tokenizer(answer_text, truncation=True, max_length=256)["input_ids"]

    # Input_ids = prompt + answer
    input_ids = prompt_ids + answer_ids
    # Labels = mask prompt tokens with -100, only keep answer tokens for loss
    labels = [-100] * len(prompt_ids) + answer_ids

    return {"input_ids": input_ids, "labels": labels}

tokenized_train = train_ds.map(tokenize_qa, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(tokenize_qa, remove_columns=val_ds.column_names)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

model_name = "meta-llama/Llama-2-7b-chat-hf"

# --- Define the quantization configuration ---
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --- Load the base model in 4-bit ---
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

# --- LoRA configuration for full attention + MLP ---
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

# --- Apply LoRA to the model ---
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Should show all attention + MLP LoRA params


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 79,953,920 || all params: 6,818,369,536 || trainable%: 1.1726


In [5]:
import wandb
wandb.login
wandb.init(project="llama3.1_custom", name="finetune_lora_3")

wandb: Currently logged in as: abhi1199 (abhi1199-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, return_tensors="pt", padding=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Perplexity will be computed in the Trainer's evaluate() output
    return {}

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

training_args = TrainingArguments(
    output_dir="./llama3_qlora_finance",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=5,
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    fp16=True,
    optim="paged_adamw_32bit",
    eval_strategy="steps",
    eval_steps=500,
    logging_dir="./logs",
    report_to="wandb"  # send metrics to W&B
)


In [9]:
from transformers import TrainerCallback, TrainerState, TrainerControl
import math

class PerplexityLoggerCallback(TrainerCallback):
    def on_evaluate(self, args, state: TrainerState, control: TrainerControl, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            eval_loss = metrics["eval_loss"]
            perplexity = math.exp(eval_loss)
            wandb.log({"eval_perplexity": perplexity, "step": state.global_step})

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[PerplexityLoggerCallback]
)


/tmp/ipython-input-1199269208.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=285, training_loss=0.27620849609375, metrics={'train_runtime': 2629.3525, 'train_samples_per_second': 1.711, 'train_steps_per_second': 0.108, 'total_flos': 2.164036231839744e+16, 'train_loss': 0.27620849609375, 'epoch': 5.0})

In [12]:
model.save_pretrained("./llama3_qlora_finance")
tokenizer.save_pretrained("./llama3_qlora_finance")
print("Finetuning complete! Model saved.")
wandb.finish()

Finetuning complete! Model saved.


train/epoch,▁▂▄▅▇█
train/global_step,▁▂▄▅▇█
train/grad_norm,█▄▃▄▁
train/learning_rate,█▆▄▃▁
train/loss,█▄▃▂▁
total_flos,2.164036231839744e+16
train/epoch,5
train/global_step,285
train/grad_norm,0.33595
train/learning_rate,3e-05
train/loss,0.0412


In [14]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

finetuned_model_path = "./llama3_qlora_finance"
base_model_name = "meta-llama/Llama-2-7b-chat-hf"

# loading the tokeinizer
tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# base model of hf
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    offload_folder="offload"
)

# base model + fine tuned model for testing
model = PeftModel.from_pretrained(base_model, finetuned_model_path)
model.eval()

print("Model loaded successfully!")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!


In [15]:

def ask_model(context, question, max_new_tokens=200, temperature=0.7):
    prompt = (
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(output[0], skip_special_tokens=True)
    # Remove the prompt from the output
    answer = answer.replace(prompt, "").strip()
    return answer


In [16]:

# test with first sample question
context = "Open Value agreements are a simple, cost-effective way to acquire the latest Microsoft technology. These agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a three-year period. Under Open Value agreements, organizations can elect to purchase perpetual licenses or subscribe to licenses and SA is included."
question = "What type of organizations is the Open Value agreements designed for and what licenses does it include?"
answer = ask_model(context, question)
print("Answer:", answer)

Answer: Open Value agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a three-year period. Under these agreements, organizations can choose to purchase perpetual licenses or subscribe to licenses, and Software Assurance (SA) is included.


In [17]:
context = "Mobility, one of the business units within the Communications segment, provides nationwide wireless service and equipment"
question = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"
answer = ask_model(context, question)
print("Answer:", answer)

Answer: Mobility


In [18]:
context = "The Company allocates the transaction price to each performance obligation on a relative SSP basis. Judgment is required to determine the SSP for each distinct performance obligation. The Company determines SSP by considering its overall pricing objectives and market conditions. Significant pricing practices taken into consideration include the Company’s discounting practices, the size and volume of the Company’s transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices."
question = "What is the basis for the Company to determine the Standalone Selling Price (SSP) for each distinct performance obligation in contracts with multiple performance obligations?"
answer = ask_model(context, question)
print("Answer:", answer)

Answer: The Company allocates the transaction price to each performance obligation on a relative SSP basis, which requires judgment on the SSP for each distinct performance obligation. This is determined by considering the overall pricing objectives and market conditions, including the Company’s discounting practices, transaction size and volume, customer demographic, geographic area, price lists, go-to-market strategy, historical and current sales and contract prices.


In [19]:
context = "As the rate implicit in the lease is rarely readily determinable, Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date."
question = "What discount rate does Delta Air Lines use for lease payments when the rate implicit in the lease is not readily determinable?"
answer = ask_model(context, question)
print("Answer:", answer)

Answer: Delta Air Lines uses the incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at the commencement date.
